# Citations over time

Which papers in this library does the field keep coming back to?

A citation count on its own says little — a 2021 paper *should* have more
citations than a 2026 one. What tells you something is the **shape of the
curve**: whether a paper is still accumulating, and how fast relative to
its age. This notebook reads the snapshot history and answers three
questions:

1. **Where things stand** — the current counts.
2. **What is moving** — how each count changed between snapshots.
3. **What is punching above its age** — citations per month since release.

All the logic lives in [`citations.py`](citations.py) next to this file;
this notebook only reads and plots. The data is `data/citations.csv`,
tracked in git, appended to (never rewritten) by:

```
python scripts/citations/citations.py
```

Run that every month or two. One snapshot gives you a ranking; several
give you a trend, which is the whole point.

> **Counts differ by source, a lot.** OpenAlex indexes the arXiv preprint
> record and often reports far fewer citations than Semantic Scholar,
> which merges the preprint with the published version. Neither is wrong.
> Compare a paper against *itself over time within one source*; never
> compare an OpenAlex number to a Semantic Scholar one.

In [ ]:
import sys
from pathlib import Path

# works whether the notebook is run from its own folder or the repo root
ROOT = Path.cwd().resolve()
while not (ROOT / "SUMMARIES.md").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts" / "citations"))

import matplotlib.pyplot as plt
import pandas as pd

import citations as C

plt.rcParams.update({"figure.figsize": (11, 6), "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})

hist = pd.DataFrame(C.load_history())
if hist.empty:
    raise SystemExit("No history yet — run  python scripts/citations/citations.py")

TITLES = {t.key: t.title for t in C.targets()}
PUBLISHED = {t.key: t.date for t in C.targets()}
hist["title"] = hist["paper"].map(TITLES).fillna(hist["paper"])
hist["published"] = hist["paper"].map(PUBLISHED)

dates = sorted(hist["date"].unique())
print(f"{hist['paper'].nunique()} papers, {len(dates)} snapshot date(s): "
      f"{dates[0]}" + (f" .. {dates[-1]}" if len(dates) > 1 else ""))
print(f"sources: {', '.join(sorted(hist['source'].unique()))}")
hist.tail(3)

## 1. Where things stand

The latest snapshot, most-cited first. `match` says how the paper was
identified: `doi` and `arxiv` are exact; `title` is a fuzzy match and is
the only one that can be wrong — check `matched_title` if a number looks
surprising.

In [ ]:
SOURCE = "semanticscholar"   # switch to "openalex" to see the conservative view

latest = (hist[(hist["source"] == SOURCE) & (hist["date"] == dates[-1])]
          .sort_values("citations", ascending=False))

latest[["published", "title", "citations", "match", "matched_title"]] \
    .reset_index(drop=True).head(50)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 12))
top = latest.head(30).iloc[::-1]
colors = ["#c44536" if m == "title" else "#38618c" for m in top["match"]]
ax.barh(top["title"].str.slice(0, 58), top["citations"], color=colors)
ax.set_xlabel(f"citations ({SOURCE}, {dates[-1]})")
ax.set_title("Most-cited papers in the library"
             "   (red = fuzzy title match, verify before quoting)")
plt.tight_layout()
plt.show()

## 2. What is moving

This is the part that needs history. With a single snapshot there is
nothing to compare, and the cell below says so instead of drawing a
misleading flat line.

The left panel is absolute counts over time; the right is the change
since the first snapshot, which is what "gaining relevance" actually
looks like.

In [ ]:
wide = (hist[hist["source"] == SOURCE]
        .pivot_table(index="date", columns="title", values="citations")
        .sort_index())

if len(wide) < 2:
    print("Only one snapshot so far — no trend to plot yet.")
    print("Re-run the snapshot script in a few weeks and this fills in.")
else:
    movers = (wide.iloc[-1] - wide.iloc[0]).sort_values(ascending=False).head(12)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 6))
    for name in movers.index:
        a1.plot(wide.index, wide[name], marker="o", label=name[:34])
    a1.set_title(f"Citation counts over time ({SOURCE})")
    a1.set_ylabel("citations")
    a1.legend(fontsize=7, loc="upper left")
    a1.tick_params(axis="x", rotation=45)

    a2.barh(movers.index.str.slice(0, 40)[::-1], movers.values[::-1],
            color="#38618c")
    a2.set_title(f"Growth since {wide.index[0]}")
    a2.set_xlabel("additional citations")
    plt.tight_layout()
    plt.show()

## 3. Punching above its age

Total citations reward age. Dividing by months since release does not —
it asks how quickly a paper accumulated attention, which is the closest
single number to "this is becoming important".

Treat it as a ranking heuristic, not a measurement. A paper three months
old has barely had time to be cited at all, so the top of this list is
noisy for anything very recent.

In [ ]:
g = pd.DataFrame(C.growth(SOURCE))
g = g[g["months_old"] >= 3]                  # too young to rank meaningfully

fig, ax = plt.subplots(figsize=(11, 10))
top = g.sort_values("per_month", ascending=False).head(25).iloc[::-1]
ax.barh(top["title"].str.slice(0, 55), top["per_month"], color="#2a9d8f")
ax.set_xlabel("citations per month since release")
ax.set_title(f"Fastest accumulation ({SOURCE}, papers at least 3 months old)")
plt.tight_layout()
plt.show()

g.sort_values("per_month", ascending=False)[
    ["published", "title", "last", "months_old", "per_month", "delta"]
].reset_index(drop=True).head(25)

In [ ]:
# The same thing as a scatter: age against total. Points above the dashed
# line accumulate faster than the library average; the far top-left is
# where a young, fast-rising paper would sit.
fig, ax = plt.subplots()
ax.scatter(g["months_old"], g["last"], s=34, color="#38618c", alpha=0.75)
avg = g["last"].sum() / g["months_old"].sum()
xs = [g["months_old"].min(), g["months_old"].max()]
ax.plot(xs, [avg * x for x in xs], "--", color="#888",
        label=f"library average ({avg:.1f}/month)")
for _, r in g.sort_values("per_month", ascending=False).head(8).iterrows():
    ax.annotate(r["title"][:26], (r["months_old"], r["last"]),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("months since release")
ax.set_ylabel(f"citations ({SOURCE})")
ax.set_title("Age vs. citations")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Do the sources agree?

They will not, and the size of the gap is worth knowing before you quote
any of these numbers in a paper. A large ratio usually means the arXiv
preprint and the published version are separate records in OpenAlex and
merged in Semantic Scholar.

In [ ]:
cmp = (hist[hist["date"] == dates[-1]]
       .pivot_table(index="title", columns="source", values="citations"))
cmp = cmp.dropna(how="any")
if cmp.shape[1] < 2:
    print("Only one source in the history — nothing to compare.")
else:
    a, b = cmp.columns[:2]
    cmp["ratio"] = (cmp[b] / cmp[a].replace(0, pd.NA)).round(1)
    display(cmp.sort_values(b, ascending=False).head(20))

    fig, ax = plt.subplots()
    ax.scatter(cmp[a], cmp[b], s=34, alpha=0.75, color="#e07a5f")
    lim = max(cmp[a].max(), cmp[b].max())
    ax.plot([0, lim], [0, lim], "--", color="#888", label="perfect agreement")
    ax.set_xlabel(a), ax.set_ylabel(b)
    ax.set_title(f"{a} vs {b}, {dates[-1]}")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Take a new snapshot

From the repository root:

```
python scripts/citations/citations.py
```

Then commit `data/citations.csv`. Appending is idempotent per day, so
re-running is harmless — but the series only exists because the file is
in git, so **the commit is the part that matters**.

Google Scholar's numbers are higher and more complete, but scraping it
breaks Google's terms of service and gets the IP CAPTCHA-blocked. It is
opt-in and prompts before running:

```
python scripts/citations/citations.py --source scholar
```